In [295]:
#Importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [296]:
# Load dataset
df = pd.read_csv("customer_shopping_behavior.csv")

# Preview data
print(df.head())

# Structure of dataset
print(df.info())

   Customer ID  Age  Gender Item Purchased     Category  \
0         2701   22  Female        T-shirt     Clothing   
1          521   51    Male     Sunglasses  Accessories   
2         3157   18  Female          Shirt     Clothing   
3         1687   22    Male         Gloves  Accessories   
4         2929   40  Female        Jewelry  Accessories   

   Purchase Amount (USD)        Location Size   Color  Season  Review Rating  \
0                   68.0      California   XL   Olive  Winter            3.2   
1                   84.0  South Carolina    M   White  Spring            3.9   
2                   50.0         Montana    M   Black  Winter            3.1   
3                   75.0        Illinois    L     Red    Fall            4.2   
4                   80.0         Alabama    L  Yellow  Spring            3.6   

  Subscription Status   Shipping Type Discount Applied  Previous Purchases  \
0                  No        Standard               No                36.0   
1       

In [297]:
# Check column names
print(df.columns)

# Summary statistics
print(df.describe())

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Previous Purchases', 'Payment Method',
       'Frequency of Purchases'],
      dtype='str')
       Customer ID          Age  Purchase Amount (USD)  Review Rating  \
count  5050.000000  5050.000000            4494.000000    4449.000000   
mean   2519.570891    44.150495             144.765236       3.668195   
std    1470.402964    15.282328             275.590101       0.865357   
min       1.000000    18.000000              10.120000       1.000000   
25%    1252.250000    31.000000              41.000000       3.000000   
50%    2499.500000    44.000000              65.000000       3.700000   
75%    3740.750000    57.000000              89.000000       4.400000   
max    5099.000000    70.000000            1499.760000       5.000000   

       Previous Pu

In [298]:
# Dataset Shape
print("Rows, Columns:", df.shape)

Rows, Columns: (5050, 17)


In [299]:
# Detect Missing Values
print(df.isnull().sum())

Customer ID                 0
Age                         0
Gender                      0
Item Purchased              0
Category                    0
Purchase Amount (USD)     556
Location                    0
Size                      370
Color                       0
Season                      0
Review Rating             601
Subscription Status         0
Shipping Type               0
Discount Applied            0
Previous Purchases        548
Payment Method              0
Frequency of Purchases      0
dtype: int64


In [300]:
# Fixing Column Names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns)

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases'],
      dtype='str')


In [301]:
df["purchase_amount_(usd)"] = pd.to_numeric(df["purchase_amount_(usd)"], errors="coerce")
df["review_rating"] = pd.to_numeric(df["review_rating"], errors="coerce")
df["previous_purchases"] = pd.to_numeric(df["previous_purchases"], errors="coerce")

In [302]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5050 non-null   int64  
 1   age                     5050 non-null   int64  
 2   gender                  5050 non-null   str    
 3   item_purchased          5050 non-null   str    
 4   category                5050 non-null   str    
 5   purchase_amount_(usd)   4494 non-null   float64
 6   location                5050 non-null   str    
 7   size                    4680 non-null   str    
 8   color                   5050 non-null   str    
 9   season                  5050 non-null   str    
 10  review_rating           4449 non-null   float64
 11  subscription_status     5050 non-null   str    
 12  shipping_type           5050 non-null   str    
 13  discount_applied        5050 non-null   str    
 14  previous_purchases      4502 non-null   float64
 15

In [303]:
#Handling Missing Values

# Calculate medians for numeric columns

median_purchase = df["purchase_amount_(usd)"].median()
median_rating = df["review_rating"].median()
median_previous = df["previous_purchases"].median()

# Fill missing values with medians

df = df.fillna({
    "purchase_amount_(usd)": median_purchase,
    "review_rating": median_rating,
    "previous_purchases": median_previous
})

# Handling Missing Values for categorical columns

df["size"] = df["size"].fillna(df["size"].mode()[0])


In [304]:
print(df.isnull().sum())

customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount_(usd)     0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
dtype: int64


In [305]:
# Removing duplicates rows
df = df.drop_duplicates() 
print("After removing duplicates, dataset shape:", df.shape)  

After removing duplicates, dataset shape: (5000, 17)


In [306]:
# Standardize text columns

text_columns = [
    "gender",
    "category",
    "location",
    "shipping_type",
    "payment_method",
    "subscription_status",
    "season",
    "discount_applied",
    "frequency_of_purchases"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.title()

In [307]:
# Removing impossible values
df = df[
    (df["purchase_amount_(usd)"] >= 0) &
    (df["review_rating"] >= 0) &
    (df["previous_purchases"] >= 0)
]

print('After cleaning, dataset shape:', df.shape)


After cleaning, dataset shape: (5000, 17)


In [308]:
# HAndling Outliers
numeric_cols = ["purchase_amount_(usd)", "review_rating", "previous_purchases", 'age']
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
    print(f'After handling outliers in {col}, dataset shape:', df.shape)

After handling outliers in purchase_amount_(usd), dataset shape: (4512, 17)
After handling outliers in review_rating, dataset shape: (4451, 17)
After handling outliers in previous_purchases, dataset shape: (4451, 17)
After handling outliers in age, dataset shape: (4451, 17)


In [309]:
# Adding a new columns for customer value based on purchase amount, loyalty, and review rating,estimate revenue potential and customer loyalty

df["customer_value"] = pd.qcut(
    df["purchase_amount_(usd)"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

df["loyal_customer"] = (
    df["previous_purchases"] >
    df["previous_purchases"].median()
)

df["estimated_revenue"] = (
    df["purchase_amount_(usd)"] *
    df["previous_purchases"]
)



In [310]:
# Final Data Validation Check

print(df.info())
print(df.describe())
print(df.isnull().sum())

<class 'pandas.DataFrame'>
Index: 4451 entries, 0 to 5049
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   customer_id             4451 non-null   int64   
 1   age                     4451 non-null   int64   
 2   gender                  4451 non-null   str     
 3   item_purchased          4451 non-null   str     
 4   category                4451 non-null   str     
 5   purchase_amount_(usd)   4451 non-null   float64 
 6   location                4451 non-null   str     
 7   size                    4451 non-null   str     
 8   color                   4451 non-null   str     
 9   season                  4451 non-null   str     
 10  review_rating           4451 non-null   float64 
 11  subscription_status     4451 non-null   str     
 12  shipping_type           4451 non-null   str     
 13  discount_applied        4451 non-null   str     
 14  previous_purchases      4451 non-null   

In [311]:
# clenaed dataset

df.to_csv("cleaned_customer_shopping_behavior1.csv", index=False)